# Model V
### Audio Spectrogram Transformer - trening sieci

Biblioteki

In [17]:
import sys
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from model import InstrumentAST ,InstrumentASTPretrained
from trening import get_device, train_model
from przygotowanie_danych import INSTRUMENTS, create_datasets, create_loaders


### Konfiguracja eksperymentu

In [18]:
# Identyfikator przygotowanego zbioru danych.
dataset = "1_3"
model_name = "AST"

epochs = 100
patience = 10
batch_size = 16
learning_rate = 0.0001

dropout = 0.3
threshold = 0.8

hidden_size = 192
num_hidden_layers = 4
num_attention_heads = 4
intermediate_size = 768

results_file = Path("wyniki/modele.csv")


### Sprawdzenie, czy eksperyment był już wykonywany

In [19]:
if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout"] == dropout) &
        (results["Hidden size"] == hidden_size) &
        (results["Transformer layers"] == num_hidden_layers) &
        (results["Attention heads"] == num_attention_heads) &
        (results["Intermediate size"] == intermediate_size)
    ).any()

    if combination_exists:
        raise ValueError(
            "Ta kombinacja datasetu i hiperparametrów "
            "była już testowana. Zatrzymuję program."
        )

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")

Nowa kombinacja. Kontynuuję wykonywanie kodu.


### Nadanie identyfikatora eksperymentu

In [20]:
results_dir = Path("wyniki")
results_dir.mkdir(parents=True, exist_ok=True)

existing_numbers = [
    int(f.stem)
    for f in results_dir.glob("*.pth")
    if f.stem.isdigit()
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

print(f"Learning ID: {learning_id}")
print(f"Plik wag: {weights_filename}")

Learning ID: 10
Plik wag: 10.pth


### Przygotowanie danych

In [21]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Pobranie przykładowego spektrogramu ze zbioru treningowego.
sample_X, sample_y = train_dataset[0]

# Automatyczne określenie wymiarów wejściowych AST.
num_mel_bins = sample_X.shape[-2]
max_length = sample_X.shape[-1]

print(f"Liczba pasm mel: {num_mel_bins}")
print(f"Długość sekwencji: {max_length}")

Wczytuję spektrogramy z: C:\spectrograms\1_3
Train: 7069
Validation: 1905
Test: 1407
Liczba pasm mel: 128
Długość sekwencji: 346


### Inicjalizacja modelu

In [22]:
device = get_device()
print(f"Urządzenie: {device}")

if model_name == "AST_pretrained":
    model = InstrumentASTPretrained(dropout=dropout).to(device)
elif model_name == "AST":
    model = InstrumentAST(
        num_mel_bins=num_mel_bins,
        max_length=max_length,
        dropout=dropout,
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        num_attention_heads=num_attention_heads,
        intermediate_size=intermediate_size
    ).to(device)

criterion = nn.MultiLabelSoftMarginLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Urządzenie: cuda


### Trening sieci

In [23]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name=weights_filename
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

Epoka 1/100 | Train loss: 0.6737 | Val loss: 0.6472


Epoka 2/100 | Train loss: 0.5726 | Val loss: 0.5410


Epoka 3/100 | Train loss: 0.5088 | Val loss: 0.4978


Epoka 4/100 | Train loss: 0.4758 | Val loss: 0.4882


Epoka 5/100 | Train loss: 0.4548 | Val loss: 0.4796


Epoka 6/100 | Train loss: 0.4356 | Val loss: 0.4937


... (9 outputs left out)

Epoka 16/100 | Train loss: 0.3290 | Val loss: 0.4823


Epoka 17/100 | Train loss: 0.3214 | Val loss: 0.4949


Epoka 18/100 | Train loss: 0.3116 | Val loss: 0.4912


Epoka 19/100 | Train loss: 0.3075 | Val loss: 0.4858
Brak poprawy przez 10 epok.
Zapisano wagi modelu: I:\Git_Projects\Projekt_Inzynierksi\instrument-recognition-project\sieci\ast\wyniki\10.pth
Najlepsza epoka: 9
Najlepszy validation loss: 0.4642
Czas treningu: 3.16 min


### Zapis informacji o eksperymencie

In [24]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout": dropout,
    "Hidden size": hidden_size,
    "Transformer layers": num_hidden_layers,
    "Attention heads": num_attention_heads,
    "Intermediate size": intermediate_size,
    "Threshold": threshold,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    results_dir / f"history_{learning_id}.csv",
    index_label="epoch"
)

print(f"Informacje o eksperymencie zapisano w: {results_file}")
print(f"Historia uczenia: history_{learning_id}.csv")

Informacje o eksperymencie zapisano w: wyniki\modele.csv
Historia uczenia: history_10.csv
